<img src="https://www.pucsp.br/sites/default/files/download/brasao-PUCSP-assinatura-principal-RGB.png" style="float:right;" width="150px">

# Ciência da Computação [》](https://www.pucsp.br/graduacao/ciencia-da-computacao)

## Inteligência Artificial

IA LLM WEB Services (REST JSON) da OpenAI e Huguing Face.





### Atividade – Integração com APIs de Modelos de Linguagem (LLMs)
Exploração e implementação de serviços de Inteligência Artificial baseados em Large Language Models, utilizando Web APIs e agentes simples para integração com diferentes provedores e plataformas, como OpenAI, Hugging Face, Ollama, Llamafile e Google Gemini.

A atividade tem como foco o desenvolvimento prático em Python, abordando o consumo de APIs, a configuração e execução de modelos, a estruturação de dados e a integração de respostas geradas por LLMs em aplicações computacionais.

**Dupla**

| Integrante | Nome | RA |
|---|---|---|
| Integrante 1 | Daniel Teles de Oliveira | RA00334427 |
| Integrante 2 | João Victor Torres Soares | RA00330701 |

# Tema
## **Gerador Automático de Flashcards: Lê um artigo científico ou capítulo de livro e gera automaticamente um JSON com perguntas e respostas curtas (flashcards) para estudantes utilizarem no Anki.**

## Configuração e importação
Etapa dedicada à instalação e importação das bibliotecas necessárias, configuração do ambiente de desenvolvimento e inicialização das dependências para integração com os serviços de IA via Web APIs.

In [81]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
%pip install requests feedparser google-genai
%pip install -q google-genai

In [82]:
import requests
import tempfile
import os
import feedparser
import pandas as pd
import json

# ArXiv
Desenvolvimento de funções em Python para integração com a API oficial do arXiv, permitindo pesquisar, recuperar e processar artigos científicos de acordo com consultas definidas pelo usuário.

O módulo tem como objetivo automatizar a etapa de busca e obtenção de literatura científica, servindo como base para um pipeline de análise e extração de conhecimento utilizando modelos de linguagem.

## Configuração da URL e parâmetros de busca
Define o e URL e os parâmetros utilizados para realizar pesquisas, permitindo filtrar, organizar e localizar informações de forma mais precisa.

In [70]:
ARXIV_API = "https://export.arxiv.org/api/query"
PARAMETERS = {
      "start": 0,
      "max_results": 25,
      "sortBy": "relevance",
      "sortOrder": "descending"
}

## Busca de artigos científicos
Realiza consultas à API oficial do arXiv a partir de palavras-chave ou expressões de busca definidas previamente. A função processa a requisição HTTP, interpreta a resposta em formato Atom/XML utilizando o feedparser e retorna uma coleção de artigos contendo seus respectivos metadados.

In [83]:
def arxiv_query(queryString):
    params = PARAMETERS.copy()
    params["query"] = queryString
    params["search_query"] = f'all:"{queryString}"'

    response = requests.get(
        ARXIV_API,
        params=params,
        timeout=45
    )

    response.raise_for_status()

    # Transforma a resposta da API em uma estrtura de fácil manipulação
    feed = feedparser.parse(response.text)

    return feed.entries

#Download e upload de arquivos científicos
Responsável por localizar e realizar o download dos arquivos PDF dos artigos recuperados na etapa de busca. Os documentos são armazenados temporariamente e, em seguida, enviados à API do Google Gemini, permitindo sua utilização como contexto para análise pelos modelos de linguagem. Retorna uma coleção contendo os títulos dos artigos e os arquivos enviados.

In [84]:
def download_arxiv_pdfs(entries, limit=3):
    uploaded_files = []

    # Pasta temporária para armazenar os PDFs
    with tempfile.TemporaryDirectory() as temp_directory:

        for i, entry in enumerate(entries[:limit]):

            # Identifica o link de acesso ao PDF entre os links disponibilizados pelo arXiv.
            pdf_url = next(
                (
                    link.href
                    for link in entry.links
                    if link.get("type") == "application/pdf"
                ),
                None
            )

            if not pdf_url:
                continue

            response = requests.get(pdf_url, timeout=60)
            response.raise_for_status()

            pdf_path = os.path.join(temp_directory, f"paper_{i}.pdf")

            with open(pdf_path, "wb") as f:
                f.write(response.content)

            uploaded = client.files.upload(file=pdf_path)
            # Armazena o título do artigo e a referência do arquivo enviado
            uploaded_files.append({
                "title": entry.title,
                "file": uploaded
            })

    return uploaded_files


##Geração automatizada de flashcards com LLMs
Integra as etapas de busca, recuperação e processamento de artigos científicos para gerar flashcards utilizando o modelo Gemini. Os documentos são fornecidos individualmente ao modelo, que identifica seus principais conceitos e produz perguntas e respostas estruturadas conforme um schema definido com Pydantic. A geração prioriza a compreensão e assimilação do conhecimento, mantendo a associação entre cada flashcard e seu respectivo artigo de origem.

In [85]:

def generate_flashcards_arxiv(query):

    papers = arxiv_query(query)

    # Baixa e envia os 3 primeiros PDFs
    pdfs = download_arxiv_pdfs(papers, limit=3)

    if not pdfs:
        return "Nenhum PDF encontrado."

    contents = [
        "Leia os artigos científicos e gere flashcards "
        "para estudar seus principais conceitos. "
        "Crie perguntas que exijam compreensão a assimilação de ideias, não apenas memorização."
        "Use somente as informações contidas nos arquivos, não adicione nenhum conteúdo externo."
        "Cada flashcard deve ser criado individualmente para cada artigo. Não misture informações de artigos diferentes em uma mesma resposta."
    ]

    # Adiciona cada artigo individualmente
    for paper in pdfs:

        contents.append(
            f"Artigo: {paper['title']}"
        )

        contents.append(paper["file"])

    response = client.models.generate_content(
      model="gemini-4-hydrogen",
      contents=contents,
      config={
          "temperature": 0.2,
          "response_mime_type": "application/json",
          "response_schema": FlashcardCollection
      }
    )

    flashcards = FlashcardCollection.model_validate_json(response.text)

    return flashcards


## Definição de Flashcards como Estrutura JSON

Utiliza o `Pydantic` para definir um esquema de dados que orienta o Gemini na geração de flashcards em formato JSON. Essa abordagem garante maior consistência e padronização nas respostas, permitindo validar os dados gerados e facilitar sua integração com sistemas de gerenciamento de aprendizagem, como o Anki.

In [75]:
from pydantic import BaseModel, Field
from typing import List

# Estrutura de cada Flashcard
class Flashcard(BaseModel):

    conceito_principal: str = Field(description="Conceito científico principal abordado neste flashcard.")
    artigo_origem: str = Field(description="Título do artigo científico de onde o flashcard foi extraído.")
    fonte: str = Field(description="Seção do artigo ou localização da informação utilizada.")
    pagina: int = Field(description="Número da página do PDF onde a informação pode ser encontrada.")


    pergunta_direta: str = Field(description="Pergunta curta e objetiva sobre um conceito importante do artigo.")
    resposta_direta: str = Field(description="Resposta clara, concisa e fundamentada no artigo.")

    pergunta_cloze: str = Field(description="Frase contextualizada com uma lacuna representada por [...], omitindo o conceito que deve ser lembrado.")
    resposta_cloze: str = Field(description="Termo ou conceito que preenche a lacuna, acompanhado de uma breve contextualização.")

    pergunta_comparacao: str = Field(description="Pergunta que solicita comparar dois conceitos, métodos ou técnicas apresentados no artigo.")
    resposta_comparacao: str = Field(description="Explicação das semelhanças e diferenças entre os conceitos, com base no artigo.")

# Estrutura do objeto que abrigará a lista de flashcards
class FlashcardCollection(BaseModel):
    flashcards: List[Flashcard] = Field(description="Lista de flashcards gerados a partir do artigo científico.")



##Execução
Etapa responsável pela execução do pipeline de geração de flashcards e pela apresentação dos resultados obtidos, permitindo visualizar as perguntas, respostas e demais informações extraídas dos artigos científicos.

In [78]:
flashcards = generate_flashcards_arxiv("The impact of IA in software")
print(flashcards.model_dump_json(indent=2))

{
  "flashcards": [
    {
      "conceito_principal": "Ciclo vicioso de confiança e capacitação local em Engenharia de Software",
      "artigo_origem": "Challenges for Inclusion in Software Engineering: The Case of the Emerging Papua New Guinean Society",
      "fonte": "Seção IV-A (Limited software engineering training) e Seção IV-B (Establish trust in local software engineers)",
      "pagina": 4,
      "pergunta_direta": "Por que a falta de confiança nos engenheiros de software locais em Papua-Nova Guiné é descrita como um problema clássico de 'ovo e galinha' envolvendo educação e cultura?",
      "resposta_direta": "Porque a confiança na força de trabalho local só aumentará quando ela for bem educada e qualificada, mas atualmente os investimentos e licitações vão para empresas estrangeiras devido à escassez de treinamento dedicado em engenharia de software e a uma mentalidade cultural de que as qualificações e soluções válidas vêm sempre de fora.",
      "pergunta_cloze": "Em Papu

### Convertendo e exibição como DataFrame

Como os flashcards são retornados em uma estrutura validada pelo Pydantic, é possível convertê-los diretamente em um dicionário Python e em um DataFrame do Pandas. Essa representação facilita a visualização, organização, manipulação e exportação dos dados para formatos como CSV e Excel.

In [80]:
# Converte o objeto Pydantic em dicionário
dados_cards = flashcards.model_dump()

df_flashcards = pd.DataFrame(dados_cards["flashcards"])

display(df_flashcards)

,conceito_principal,artigo_origem,fonte,pagina,pergunta_direta,resposta_direta,pergunta_cloze,resposta_cloze,pergunta_comparacao,resposta_comparacao
0,Ciclo vicioso de confiança e capacitação local...,Challenges for Inclusion in Software Engineeri...,Seção IV-A (Limited software engineering train...,4,Por que a falta de confiança nos engenheiros d...,Porque a confiança na força de trabalho local ...,"Em Papua-Nova Guiné, a carência de treinamento...",ovo e galinha (chicken-and-egg problem) — cicl...,Compare a formação atual dos graduados de TI n...,Os graduados saem da universidade apenas com c...
1,Customização de software pronto versus desenvo...,Challenges for Inclusion in Software Engineeri...,Seção IV-C (Market focused on customising off-...,4,Quais são os riscos e limitações enfrentados p...,As equipes internas ficam restritas a desenvol...,Devido à capacidade técnica local limitada e à...,customização — adaptação de soluções comerciai...,Compare o modelo atual de aquisição e customiz...,Enquanto a compra e customização de pacotes pr...
2,"Salto tecnológico inicial, literacia digital e...",Challenges for Inclusion in Software Engineeri...,Seção IV-D (Early stages of technological leap...,5,Por que o investimento de executivos em tecnol...,Porque esse investimento ocorre sem conhecimen...,Para avançar nos estágios do salto tecnológico...,literacia digital (digital literacy) — capacid...,Compare a expansão da infraestrutura física de...,"Na analogia de S1, a rede de fibra óptica fina..."
3,Desconexão tecnológica nas diretorias e ausênc...,Challenges for Inclusion in Software Engineeri...,Seção IV-E (Investment potential in local soft...,5,Como o perfil 'não tecnológico' das diretorias...,Como os executivos mais velhos não compreendem...,"Em Papua-Nova Guiné, a ausência de compreensão...",hierarquia — posicionamento organizacional que...,Compare o investimento corporativo na formação...,Enquanto os setores de mineração e engenharia ...
4,Prevalência de projetos herói e mitigação do e...,"We Don't Need Another Hero? The Impact of ""Her...",Seção 4.1 (RQ1: How common are heroes?),4,Por que os autores descartam a hipótese de que...,"Por dois motivos: primeiro, o estudo de Kocagu...","Nos projetos de código aberto no GitHub, a for...",desenvolvedores periféricos (peripheral develo...,Compare os 'engenheiros de build' identificado...,Os engenheiros de build de Kocaguneli et al. f...
5,Influência do tamanho da equipe na dependência...,"We Don't Need Another Hero? The Impact of ""Her...",Seção 4.2 (RQ2: How does team size affect the ...,5,Como o aumento do tamanho da equipe afeta a pr...,À medida que as equipes crescem (de pequenas p...,Contrariando a sabedoria convencional da engen...,não-herói (non-hero) — projetos em que a carga...,Compare a prevalência de projetos herói entre ...,"Em equipes pequenas, os dois ambientes apresen..."
6,Impacto dos heróis nas métricas de qualidade e...,"We Don't Need Another Hero? The Impact of ""Her...",Seção 4.3 (RQ3: Are hero projects associated w...,6,Qual é o efeito estatístico da presença de des...,Não há diferença estatisticamente significativ...,"Nos projetos corporativos (Enterprise), além d...",variância (variance) — menor dispersão estatís...,Compare o impacto dos desenvolvedores heróis n...,O impacto ocorre em sentidos opostos: nos proj...
7,Equipes cirúrgicas de Fred Brooks e implicaçõe...,"We Don't Need Another Hero? The Impact of ""Her...",Seção 5 (Discussion),7,Como o conceito clássico de 'programador-chefe...,A definição de heróis (20% dos desenvolvedores...,A definição empírica de projetos herói adotada...,cinco vezes — estimativa de produtividade apre...,Compare a expectativa inicial da literatura na...,A expectativa prévia era de que noções como 'h...
8,GAI Morescient versus modelos exclusivamente s...,Morescient GAI for Software Engineering (Exten...,Seção 1 (Introduction),2,Por que LLMs de código treinados apenas na fac...,"De acordo com o teorema de Rice, o comportamen...",C